# Figure: Lorenz System — EDMD with Time-Delay Embedding

**System:** $\dot{x}=\sigma(y-x),\ \dot{y}=x(\rho-z)-y,\ \dot{z}=xy-\beta z$  
($\sigma=10,\rho=28,\beta=8/3$)

**Dictionary:** Time-delay embedding, $N_{\text{delay}}=100$ delays $\times$ 3 coordinates $= 300$ observables.

**Outputs:** Pseudo-eigenfunction visualizations on the Lorenz attractor (dark background).  
Comparison grid: rows = method (Incremental / PageRank / PPR), columns = $N \in \{30,60,120,300\}$.


In [ ]:
import sys, os
sys.path.insert(0, '../src')

import numpy as np
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D
from copy import deepcopy
import koopman_source as kp

plt.rcParams.update({
    "font.family": "sans-serif",
    "mathtext.fontset": "cm",
    "font.size": 9,
    "axes.labelsize": 10,
    "axes.titlesize": 11,
    "figure.dpi": 150,
    "savefig.dpi": 400,
})

COLOR_PR  = '#2f9f47'
COLOR_PPR = '#e377c2'
COLOR_Inc = '#9467bd'

SAVE_FIGS = False
FIG_DIR   = 'figures/'
os.makedirs(FIG_DIR, exist_ok=True)

elev0 = 20; azim0 = -50
print('Setup done.')


## Lorenz trajectory generation (RK4)

In [ ]:
sigma_lor = 10.0
rho_lor   = 28.0
beta_lor  = 8.0 / 3.0
dt_sim    = 0.001   # fine RK4 step
dt_delay  = 0.2     # delay stride
h_skip    = int(dt_delay / dt_sim)

M_lor     = 200_000  # training snapshot pairs
n_warmup  = 100_000  # fine steps to reach attractor
N_delay   = 100      # delay levels  ->  N_lor = 300

np.random.seed(1234)
x_cur = (np.random.rand(3) - 0.5) * 4.0
for _ in range(n_warmup):
    x_cur = kp.lorenz_rk4(x_cur, dt_sim, sigma_lor, rho_lor, beta_lor)

n_fine     = M_lor + h_skip * N_delay + 2
traj_fine  = np.zeros((3, n_fine + 1))
traj_fine[:, 0] = x_cur
for i in range(n_fine):
    traj_fine[:, i + 1] = kp.lorenz_rk4(traj_fine[:, i], dt_sim,
                                          sigma_lor, rho_lor, beta_lor)

print(f'Fine trajectory shape: {traj_fine.shape}')
print(f'x: [{traj_fine[0].min():.1f}, {traj_fine[0].max():.1f}]')
print(f'y: [{traj_fine[1].min():.1f}, {traj_fine[1].max():.1f}]')
print(f'z: [{traj_fine[2].min():.1f}, {traj_fine[2].max():.1f}]')


## Time-delay embedding & full EDMD

In [ ]:
N_lor        = 3 * N_delay
T_total      = traj_fine.shape[1]
M_train_lor  = T_total - h_skip * N_delay - 1

PsiX_lor, PsiY_lor = kp.build_delay_embedding(traj_fine, M_train_lor, N_delay, h_skip)
coord_lor = np.array([0, 1, 2], dtype=int)   # current-time x, y, z
other_lor = np.setdiff1d(np.arange(N_lor), coord_lor)

K_lor = kp.find_best_lin(PsiX_lor, PsiY_lor)

print(f'N_delay={N_delay}, h_skip={h_skip}, N_lor={N_lor}')
print(f'Training pairs M_train_lor = {M_train_lor:,}')
print(f'K_lor shape: {K_lor.shape}')


## PageRank / PPR orderings

In [ ]:
GROUP_BY_DELAY = False   # True = group x/y/z per delay level

pr_v_lor  = kp.pagerank(np.abs(K_lor), num_iterations=100_000, d=0.85)
ppr_v_lor = kp.personalized_pagerank(np.abs(K_lor), num_iterations=100_000,
                                       d=0.85, seed_nodes=list(coord_lor))
np.random.seed(123)
random_idx_lor = np.concatenate([coord_lor, np.random.permutation(other_lor)]).astype(int)
inc_v_lor = np.arange(N_lor, dtype=float)[::-1].copy()

methods_cfg = {
    'PPR':         ppr_v_lor,
    'PageRank':    pr_v_lor,
    'Incremental': inc_v_lor,
}
print('Scores computed.')
print(f'GROUP_BY_DELAY = {GROUP_BY_DELAY}')


## Eigenfunction selection helpers

In [ ]:
def select_eigenfunctions(vals, vecs, Psi_plot, sel_indices,
                           n_ef=4, angle_offset=0.006,
                           trivial_mod_tol=1e-4, trivial_angle_tol=1e-3,
                           variation_ratio=0.05):
    angles      = np.abs(np.abs(np.angle(vals)) - angle_offset)
    order       = np.argsort(angles)
    vals_sorted = vals[order]
    vecs_sorted = vecs[:, order]
    ef_indices  = []
    phi_list    = []
    for i in range(len(vals_sorted)):
        lam = vals_sorted[i]
        if (np.abs(np.abs(lam) - 1.0) < trivial_mod_tol and
                np.abs(np.angle(lam)) < trivial_angle_tol):
            continue
        v   = np.real(vecs_sorted[:, i])
        phi = np.real(Psi_plot[:, sel_indices] @ v)
        if np.std(phi) / (np.abs(np.mean(phi)) + 1e-12) > variation_ratio:
            ef_indices.append(i)
            phi_list.append(phi)
        if len(ef_indices) >= n_ef:
            break
    return vals_sorted, vecs_sorted, ef_indices, phi_list


def match_to_reference(vals_sub, target_angles):
    angles_sub = np.angle(vals_sub)
    matched = []
    used    = set()
    for tgt in target_angles:
        diffs = np.abs(angles_sub - tgt)
        for idx in np.argsort(diffs):
            if idx not in used:
                matched.append(int(idx))
                used.add(int(idx))
                break
    return matched

print('Helpers defined.')


## Compute reference eigenfunctions (Full basis, N=300)

In [ ]:
traj_plot = traj_fine[:, :M_train_lor]
Psi_plot  = PsiX_lor

vals_full_raw, vecs_full_raw = np.linalg.eig(K_lor)
sel_full = np.arange(N_lor, dtype=int)

_, _, ef_full, phi_full = select_eigenfunctions(
    vals_full_raw, vecs_full_raw, Psi_plot, sel_full, n_ef=1
)

angles_full_raw  = np.abs(np.abs(np.angle(vals_full_raw)) - 0.006)
order_full       = np.argsort(angles_full_raw)
vals_full_sorted = vals_full_raw[order_full]
target_angles    = np.angle(vals_full_sorted[ef_full])

print(f'Full basis: {len(ef_full)} reference eigenfunctions')
print(f'Reference eigenvalue angles: {np.round(target_angles, 4)}')


## Compute eigenfunctions for all methods & subset sizes

In [ ]:
N_select_list = [300, 120, 60, 30]
cmaps_lor     = ['RdBu_r', 'RdBu_r', 'RdBu_r', 'RdBu_r']
results       = {}

for method_name, rank_vec in methods_cfg.items():
    print(f'\nMethod: {method_name}')
    for N_sel in N_select_list:
        if N_sel == N_lor:
            sel = np.arange(N_lor, dtype=int)
        else:
            sel = kp.select_top_indices(rank_vec, coord_lor, N_delay, N_sel,
                                         group_by_delay=GROUP_BY_DELAY)

        K_sub = kp.find_best_lin(PsiX_lor[:, sel], PsiY_lor[:, sel])
        vals_sub, vecs_sub = np.linalg.eig(K_sub)
        matched_idx  = match_to_reference(vals_sub, target_angles)
        vals_matched = vals_sub[matched_idx]
        phi_matched  = [np.real(Psi_plot[:, sel] @ np.real(vecs_sub[:, i]))
                        for i in matched_idx]
        results[(method_name, N_sel)] = {'sel': sel, 'vals': vals_matched, 'phis': phi_matched}
        print(f'  N_sel={N_sel}: |λ|={[f"{abs(v):.4f}" for v in vals_matched]}')

print('\nAll eigenfunctions computed.')


## Comparison grid: rows = method, columns = N

In [ ]:
N_select_list_fig = N_select_list[::-1]   # [30, 60, 120, 300]
methods_fig       = ['Incremental', 'PageRank', 'PPR']
cmap_fig          = 'RdBu_r'

n_rows = len(methods_fig)
n_cols = len(N_select_list_fig)

fig, axes = plt.subplots(
    n_rows, n_cols,
    figsize=(n_cols * 3.2, n_rows * 2.6),
    facecolor='white',
    subplot_kw={'projection': '3d'},
    gridspec_kw={'wspace': -0.3, 'hspace': -0.25}
)

for row, method_name in enumerate(methods_fig):
    for col, N_sel in enumerate(N_select_list_fig):
        ax  = axes[row, col]
        key = (method_name, N_sel)
        ax.set_facecolor('white')
        if key not in results:
            ax.axis('off')
            continue

        vals_m   = results[key]['vals']
        phi_list = results[key]['phis']
        ef_idx   = 0
        lam      = vals_m[ef_idx]
        phi      = phi_list[ef_idx]
        w_c      = np.log(lam + 1e-30) / dt_sim

        phi_n = phi / (np.linalg.norm(phi) + 1e-12)
        vmax  = 3.0 * np.mean(np.abs(phi_n))

        ax.scatter(traj_plot[0], traj_plot[1], traj_plot[2],
                   c=phi_n, cmap=cmap_fig, vmin=-vmax, vmax=vmax,
                   s=0.3, alpha=0.9, rasterized=True, edgecolors='none')

        for pane in [ax.xaxis.pane, ax.yaxis.pane, ax.zaxis.pane]:
            pane.fill = False
            pane.set_edgecolor('none')
        ax.grid(False)
        ax.set_xticklabels([]); ax.set_yticklabels([]); ax.set_zticklabels([])
        ax.set_axis_off()
        ax.view_init(elev=25, azim=-40)

        if row == 0:
            ax.set_title(rf'$N = {N_sel}$', fontsize=13, pad=-10, color='black')

    axes[row, 0].text2D(
        -0.05, 0.5, method_name,
        transform=axes[row, 0].transAxes,
        fontsize=12, va='center', ha='right', rotation=90, color='black'
    )

w_ref = np.log(results[('PPR', 30)]['vals'][0] + 1e-30) / dt_sim
plt.suptitle(
    rf'Lorenz eigenfunctions ($\log(\lambda)/(i\Delta t) \approx {w_ref.imag:.1f}$ rad/s)',
    fontsize=14, y=0.98, color='black'
)

if SAVE_FIGS:
    plt.savefig(f'{FIG_DIR}lorenz_comparison_grid.pdf',
                dpi=400, bbox_inches='tight', facecolor='white')
plt.show()
